In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

from run_epoch import run_epoch, run_epoch_torch, get_vit_model, get_loaders
from utils import seed_everything, Settings

In [3]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [12]:
print(torch.cuda.memory_reserved() / 1024**3)
print(torch.cuda.memory_allocated() / 1024**3)

0.26953125
0.24262571334838867


### 0. Default

In [7]:
def setup():
    seed_everything()
    model = get_vit_model()
    train_loader, val_loader = get_loaders()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=Settings.lr)
    return model, train_loader, val_loader, criterion, optimizer

In [ ]:
run_epoch(*setup())

In [ ]:
run_epoch_torch(*setup())

Train:   6%|▌         | 1/17 [00:45<12:01, 45.09s/it]


### 1. Fix dataloader

In [ ]:
# pin memory=True, no CPU overhead in cudaMemcpyAsync
# from 52ms to 2ms in cudaMemcpyAsync
run_epoch_torch(*setup())

Train:   6%|▌         | 1/17 [00:59<15:45, 59.08s/it]


In [ ]:
# num_workers=4, from 2ms to 56us in CPU cudaMemcpyAsync
run_epoch_torch(*setup())

Dataset already extracted
Train Data: 4322
Val Data: 1081


Train:   6%|▌         | 1/17 [00:49<13:04, 49.05s/it]


### 2. Hidden dim 255 to 256

In [10]:
run_epoch_torch(*setup())

Dataset already extracted
Train Data: 4322
Val Data: 1081


Train:   6%|▌         | 1/17 [00:43<11:41, 43.83s/it]


In [12]:
run_epoch(*setup())

Dataset already extracted
Train Data: 4322
Val Data: 1081
Using custom schedule function


Train:   6%|▌         | 1/17 [00:28<07:32, 28.29s/it]

Step 1, Phase: active
Step 2, Phase: inactive


Train:   6%|▌         | 1/17 [00:43<11:35, 43.46s/it]


### 3. QKV in one mm

In [19]:
torch.backends.cuda.enable_flash_sdp(True)
torch.backends.cuda.enable_math_sdp(True)
torch.backends.cuda.enable_mem_efficient_sdp(True)
# qkv = self.to_qkv(self.norm(x))
run_epoch_torch(*setup())

Dataset already extracted
Train Data: 4322
Val Data: 1081


Train:   6%|▌         | 1/17 [00:48<13:01, 48.81s/it]


# 4 and 5. Compile and fp16

In [26]:
seed_everything()
model = get_vit_model()
train_loader, val_loader = get_loaders()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=Settings.lr)

Dataset already extracted
Train Data: 4322
Val Data: 1081


In [27]:
from run_epoch import run_epoch_torch_fp16
run_epoch_torch_fp16(model, train_loader, val_loader, criterion, optimizer)

Train:   6%|▌         | 1/17 [00:49<13:06, 49.16s/it]
